The plan for this file it to have all the code to go from the different planet spectra to them making a new planet list hopefully in an organsied way so it can all be done from one file. 


In [ ]:
#--- Imports ---#
import matplotlib.pyplot as plt
import numpy as np 
import pandas as pd
from uncertainties import  ufloat
from scipy import constants
from pathlib import Path
from typing import Tuple, List
from functools import partial
from time import time
import concurrent.futures
from tqdm import tqdm
import os
from datetime import datetime 

#--- Data sheets ---# 

Planet_data= pd.read_csv('Data/planets.csv', comment='#')



# --- Classes ---#

class Configuration:








In [ ]:
#-----------------------------------------Abis cell-----------------------------

#---reading in the file from the github data

!pip install pandas
import pandas as pd
import numpy as np

#read in file path for data from the github
#path = "C:/Users/abiga/OneDrive - University of Birmingham/Documents/year 3 - semester 2/Labs/code/results20260205_160505.csv"
path = "C:/Users/abiga/OneDrive - University of Birmingham/Documents/year 3 - semester 2/Labs/code/results20260205_175059.csv"
data = pd.read_csv(path)

data.columns = (
    data.columns
        .str.strip()      # remove leading/trailing spaces
        .str.lstrip('#')  # remove leading #
)

#causing problems with the column names, chat suggested this and it fixed the problem
#dont forget to put a space between ' and file name

print(data.columns)

exo_data = data[[' File_Name', 'Equilibrium_Temperature',
       'Uncertainty_on_Equilibrium_Temperature', 'A_H', 'A_H_error',
       'Number_of_transits']]

#-------All model definitions and runnning MCMC

import matplotlib.pyplot as plt
import emcee

eq_temp = np.asarray(data["Equilibrium_Temperature"])
A_H = np.asarray(data["A_H"])
A_H_err = np.asarray(data["A_H_error"])

mask = np.isfinite(eq_temp) & np.isfinite(A_H) & np.isfinite(A_H_err)
eq_temp = eq_temp[mask]
A_H = A_H[mask]
A_H_err = A_H_err[mask]

idx = np.argsort(eq_temp)
#for a clean line
x_line = eq_temp[idx]
y_line = A_H[idx]
yerr_line = A_H_err[idx]

x0 = np.mean(eq_temp)
xs = np.std(eq_temp)
x_s = (eq_temp - x0) / xs


def linear_model(theta, x_s):          #linear model
    m, b = theta
    return m*x_s + b

def quad_model(theta, x_s):            #quadratic model
    a, b, c = theta
    return a*x_s**2 + b*x_s + c

def ln_likelihood(model, theta, x_s, y, yerr):       #log liklihood
    y_model = model(theta, x_s)
    inv_sigma2 = 1.0/(yerr**2)
    return -0.5*np.sum((y - y_model)**2 * inv_sigma2 + np.log(2*np.pi*yerr**2))

def ln_prior_linear(theta):      #linear priors
    m, b = theta
    if -50 < m < 50 and -50 < b < 50:
        return 0.0
    return -np.inf

def ln_prior_quad(theta):       #quadrtatic priors
    a, b, c = theta
    # with scaled x, these ranges are usually sensible
    if -50 < a < 50 and -50 < b < 50 and -50 < c < 50:
        return 0.0
    return -np.inf

def ln_prob_linear(theta, x_s, y, yerr):    #linear log probability
    lp = ln_prior_linear(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + ln_likelihood(linear_model, theta, x_s, y, yerr)

def ln_prob_quad(theta, x_s, y, yerr):      #quadratic log probability
    lp = ln_prior_quad(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + ln_likelihood(quad_model, theta, x_s, y, yerr)

#------------ MCMC

import emcee

def run_mcmc(ln_prob, ndim, p0, nwalkers=32, burn=1000, steps=2000):
    sampler = emcee.EnsembleSampler(nwalkers, ndim, ln_prob, args=(x_s, A_H, A_H_err))
    state = sampler.run_mcmc(p0, burn, progress=True)
    sampler.reset()
    sampler.run_mcmc(state, steps, progress=True)
    return sampler


nwalkers = 32

# linear
ndim_lin = 2
p0_lin_center = np.array([0.0, 0.0])
p0_lin = p0_lin_center + 1e-2*np.random.randn(nwalkers, ndim_lin)
sampler_lin = run_mcmc(ln_prob_linear, ndim_lin, p0_lin)

# quadratic
ndim_quad = 3
p0_quad_center = np.array([0.0, 0.0, 0.0])
p0_quad = p0_quad_center + 1e-2*np.random.randn(nwalkers, ndim_quad)
sampler_quad = run_mcmc(ln_prob_quad, ndim_quad, p0_quad)

import matplotlib.pyplot as plt

x_line = np.linspace(eq_temp.min(), eq_temp.max(), 300)
x_line_s = (x_line - x0)/xs

# sample a few posterior draws
s_lin = sampler_lin.get_chain(flat=True)
s_quad = sampler_quad.get_chain(flat=True)

plt.figure()
#plt.plot(x_line, y_med, color="navy", lw=3, label="Median fit")
plt.errorbar(eq_temp, A_H, yerr=A_H_err, fmt="o", alpha=0.4, label="data", color = 'slategray')

for theta in s_lin[np.random.randint(len(s_lin), size=50)]:
    plt.plot(x_line, linear_model(theta, x_line_s), alpha=0.15, color = 'blue')

#for theta in s_quad[np.random.randint(len(s_quad), size=50)]:
#    plt.plot(x_line, quad_model(theta, x_line_s), alpha=0.15, color = 'purple')

plt.xlabel("x")
plt.ylabel("A_H")
plt.ylim(-100,200)
plt.title("Linear model")
plt.legend()
plt.show()

#plt.plot(x_line, y_med_quad, color="darkred", lw=3, label="Median fit")
plt.errorbar(eq_temp, A_H, yerr=A_H_err, fmt="o", alpha=0.4, label="data", color = 'slategray')
for theta in s_quad[np.random.randint(len(s_quad), size=50)]:
    plt.plot(x_line, quad_model(theta, x_line_s), alpha=0.15, color = 'purple')

plt.xlabel("x")
plt.ylabel("A_H")
plt.title("Quadratic model")
plt.ylim(-100, 200)
plt.legend()
plt.show()

#-------------------BIC calculation

theta_lin = [0.5, 20]
theta_quad = [-0.05, -2, 13]

lnL_lin = ln_likelihood(linear_model, theta_lin, eq_temp, A_H, A_H_err)
lnL_quad = ln_likelihood(quad_model, theta_quad, eq_temp, A_H, A_H_err)

k_lin = 2
k_quad = 3
n = len(A_H)

BIC_lin = k_lin * np.log(n) - 2 * (lnL_lin)
print(f"linear BIC value:", BIC_lin)
BIC_quad = k_quad * np.log(n) - 2 * (lnL_quad)
print(f"quadratic BIC value:",BIC_quad)   
delta_BIC = BIC_quad - BIC_lin
print("-------")
print(f"delta BIC:", delta_BIC)

best_BIC = "Linear" if BIC_lin < BIC_quad else "Quadratic"
print("-------")
print(best_BIC, f"is the model to be favoured")

#relative likelihood
rel_like = np.exp(-0.5*delta_BIC)
print(f"the relative likelihood is:", rel_like)
#this should be fine but BIC is very large

#----------------corner plots, autocorrection time and liklihood values

#linear 

import corner

labels_lin = ["m", "b"]

flat_samples_lin = sampler_lin.get_chain(discard=1000, thin=1, flat=True)

fig = corner.corner(
    flat_samples_lin,
    labels=labels_lin,
    show_titles=True
)

imax = np.argmax(flat_samples_lin)

m_ml, b_ml = flat_samples_lin[imax]
#s_int_ml = np.exp(log_s_ml)

print("Best sample (MAP):")
print("m =", m_ml)
print("b =", b_ml)
#print("log_s =", log_s_ml)
#print("sigma_int =", s_int_ml)


print("----")
#autocorrection time
tau = sampler_lin.get_autocorr_time()
print(f"autocorrection time is:")
print(tau)
#flat_samples = sampler_lin.get_chain(discard=1000, thin=1, flat=True)
print(flat_samples_lin.shape)

# quadratic
import numpy as np
import corner

labels_quad = ["a", "b", "c"]

# chain and log-prob, flattened the same way
flat_samples_quad = sampler_quad.get_chain(discard=1000, thin=1, flat=True)
flat_logprob_quad = sampler_quad.get_log_prob(discard=1000, thin=1, flat=True)

fig = corner.corner(
    flat_samples_quad[:, :3],   # in case you ever have extra params later
    labels=labels_quad,
    show_titles=True
)

# MAP / "best" sample = highest posterior (log-prob)
imax_Q = np.argmax(flat_logprob_quad)
a_ml, b_ml, c_ml = flat_samples_quad[imax_Q, :3]

print("Best sample (MAP):")
print("a =", a_ml)
print("b =", b_ml)
print("c =", c_ml)

print("----")

#--------------autocorrelation time (this can fail if chain is too short; handle gracefully)
try:
    tau = sampler_quad.get_autocorr_time()
    print("autocorrelation time is:")
    print(tau)
except Exception as e:
    print("autocorr time couldn't be estimated reliably (chain may be too short).")
    print("Error:", e)

print("flat_samples_quad shape:", flat_samples_quad.shape)
print("flat_logprob_quad shape:", flat_logprob_quad.shape)

#--------------posterior fits

plt.figure(figsize=(8,5))

# x grid for smooth lines (RAW axis)
x_grid = np.linspace(eq_temp.min(), eq_temp.max(), 300)
x_grid_s = (x_grid - x0) / xs   # scaled version for the model

inds = np.random.randint(len(flat_samples_lin), size=300)

for ind in inds:
    m, b = flat_samples_lin[ind][:2]
    y_line = m * x_grid_s + b
    plt.plot(x_grid, y_line, color="steelblue", alpha=0.10, lw=1)

plt.errorbar(eq_temp, A_H, yerr=A_H_err, fmt=".", capsize=0,
             color="slategray", alpha=0.35, label="data")

plt.xlabel("eq_temp")
plt.ylabel("A_H")
plt.title("Linear posterior fits")
plt.ylim(-100,200)
plt.legend()
plt.show()

indsQ = np.random.randint(len(flat_samples_quad), size=300)

for ind in indsQ:
    m, b = flat_samples_quad[ind][:2]
    y_line = a * (x_grid_s)**2 + b*(x_grid_s) + c
    plt.plot(x_grid, y_line, color="steelblue", alpha=0.10, lw=1)

plt.errorbar(eq_temp, A_H, yerr=A_H_err, fmt=".", capsize=0,
             color="slategray", alpha=0.35, label="data")

plt.xlabel("eq_temp")
plt.ylabel("A_H")
plt.title("Quadratic posterior fits")
plt.ylim(-100,200)
plt.legend()
plt.show()

#------------(reduced) chi squared

import numpy as np

# theta_lin: [m, b] or [m, b, log_s]
# theta_quad: [a, b, c] or [a, b, c, log_s]
# models take (theta, x_s) and return y_model

def chi_squared(model, theta, x_s, y, yerr):
    y_model = model(theta, x_s)
    return np.sum(((y - y_model) / yerr)**2)

def reduced_chi_squared(model, theta, x_s, y, yerr, n_params):
    chi2 = chi_squared(model, theta, x_s, y, yerr)
    dof = len(y) - n_params
    return chi2, chi2 / dof

# Use your actual data arrays
y = A_H
yerr = A_H_err

# number of fitted parameters (exclude log_s if you’re using it as a nuisance parameter)
k_lin  = 2
k_quad = 3

chi2_lin, red_chi2_lin   = reduced_chi_squared(linear_model, theta_lin,  x_s, y, yerr, k_lin)
chi2_quad, red_chi2_quad = reduced_chi_squared(quad_model,   theta_quad, x_s, y, yerr, k_quad)

print("linear chi2:", chi2_lin)
print("quadratic chi2:", chi2_quad)
print("reduced chi2 (linear, quad):", red_chi2_lin, red_chi2_quad)

# Bar plot for BIC and reduced chi-squared
fig, ax1 = plt.subplots(figsize=(6,4))

# BIC
ax1.bar(['Linear', 'Quadratic'], [BIC_lin, BIC_quad], color=['skyblue','salmon'], alpha=0.7)
ax1.set_ylabel("BIC", color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

# Overlay reduced chi-squared on second axis
ax2 = ax1.twinx()
ax2.bar(['Linear', 'Quadratic'], [red_chi2_lin, red_chi2_quad], color=['lightgreen','orange'], alpha=0.4)
ax2.set_ylabel("Reduced χ²", color='green')
ax2.tick_params(axis='y', labelcolor='green')

plt.title("Comparison: BIC vs Reduced χ²")
#plt.ylim(0, 2e8)
plt.show()